In [ ]:
import os
import wave
import librosa
from scipy.io import wavfile
import noisereduce as nr
import itertools
import numpy as np
import IPython.display as ipd
from pydub import AudioSegment,silence
import librosa.display
import matplotlib.pyplot as plt
import wave
import numpy
import pandas as pd
import statistics
from scipy.stats import variation
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
import matplotlib.ticker as mticker

In [ ]:
# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory

In [ ]:
# IMPORTANT:
# Create a folder named 'Visualize' inside base_dir.
# Copy/save the audio (.wav) files that need visualization into this folder.
# The script expects files in:
# base_dir + '/Visualize/' + par_code + '.wav'

In [ ]:
%run -i "$base_dir/Scripts/ATAS_functions_CWS_CWNS.ipynb" # Change to required destination

In [ ]:
pause_time_threshold = 50
speech_time_threshold = 100
segment_size_t = 1 # smaller segments to calculate mean within longer segment_size_t_windows (if required)
segment_size_t_window = 3 # segment size in seconds

## CWNS/ CWS Visualization

In [ ]:
def viz_S_P_A(trim_file_viz, ddf):
    x_nr_ex, sr_nr_ex = librosa.load(trim_file_viz, sr=None)

    # %matplotlib notebook   # run once in notebook, not inside function
    plt.figure(figsize=(12, 5))

    # --- Plot event spans ---
    for i in ddf.index:
        start = ddf.Time_start[i]
        end = ddf.Time_end[i]

        if (ddf["Labels"][i] == 'p') and (ddf["Time_diff"][i] > 0.15):
            plt.axvspan(start, end, color='r', alpha=0.2)
        if (ddf["Labels"][i] == 'p') and (ddf["Time_diff"][i] <= 0.15):
            plt.axvspan(start, end, color='blue', alpha=0.2)
        if ddf["Labels"][i] == 's':
            plt.axvspan(start, end, color='g', alpha=0.2)

    # --- Legend ---
    red_patch = mpatches.Patch(color='r', alpha=0.2, label='Long Pause (>150 ms)')
    blue_patch = mpatches.Patch(color='blue', alpha=0.2, label='Short Pause (<=150 ms)')
    green_patch = mpatches.Patch(color='g', alpha=0.2, label='Vocal Event (>100 ms)')
    plt.legend(handles=[red_patch, blue_patch, green_patch])

    # --- Waveform ---
    librosa.display.waveshow(x_nr_ex, sr=sr_nr_ex)

    ax = plt.gca()

    # --- Exact start/end from ddf ---
    start_time = float(ddf["Time_start"].min())
    end_time = float(ddf["Time_end"].max())

    # Lock x-axis exactly
    ax.set_xlim(start_time, end_time)

    # Get default ticks, keep in range, and force exact endpoints
    base_ticks = ax.get_xticks()
    ticks = base_ticks[(base_ticks >= start_time) & (base_ticks <= end_time)]
    ticks = np.unique(np.concatenate([ticks, [start_time, end_time]]))

    ax.set_xticks(ticks)

    # Format all ticks to 3 decimals (so 10.555 displays exactly)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

    plt.xlabel('Time', fontsize=15) # Relabel with appropriate unit
    plt.ylabel('Amplitude', fontsize=15)

    plt.tight_layout()


In [ ]:
# -------------------------------------------------------------------------
# Example usage:
# This section demonstrates how to:
# 1. Load and preprocess a selected audio file
# 2. Run segmentation and event detection
# 3. Visualize detected speech and pause events
# -------------------------------------------------------------------------

In [ ]:
par_code = "example_participant_id" # Include part of the audio file name to plot (e.g. Participant ID)
file_name = base_dir + '/Visualize/'+ par_code + '.wav'


# --- Set times + original filename ---
start_time = 0.000
end_time = 8.170 # The closet last event detected to this time point will be the last in display
#end_time = librosa.get_duration(path=file_name)


file_name_org = file_name
outfile_name = par_code+'_proc_nr_norm.wav'
print(outfile_name)
os.chdir(os.path.dirname(base_dir))
x_nr_ex, sr_nr_ex, trim_file, f_trim_file, trim_file_viz, f_name_2 = wav_file_trim_wf(
    outfile_name, start_time, end_time
)
all_seg_details_silence_p_file, all_seg_details_silence_rem_file, seg_time_leng = event_detection_seg_wf_window(x_nr_ex, sr_nr_ex, segment_size_t_window, trim_file)            
silence_p_final_end, silence_rem_final_end,ddf,long_p,short_p,long_p_durations,short_p_durations = detect_proper_events(all_seg_details_silence_p_file,all_seg_details_silence_rem_file,speech_time_threshold,pause_time_threshold, file_name, seg_time_leng)


In [ ]:
# Visualize detection
viz_S_P_A (trim_file_viz,ddf)
plt.xlabel('Time', fontsize=15)  # Re-setting x-axis label # Relabel with appropriate unit
plt.show()